# 01 - Data & preprocessing
What the three datasets look like, and exactly how they are prepared. Key rule: **split first, fit preprocessing on train only**.

In [1]:
import sys; sys.path.insert(0, '../src')
import pandas as pd
from arena.data import load_titanic, load_california, load_mall
from arena.evaluate import split
from arena.preprocess import build_preprocessor

## Titanic (classification)

In [2]:
X, y = load_titanic()
print(X.shape, 'survival rate = %.3f' % y.mean())
X.head()

(891, 10) survival rate = 0.384


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize,IsAlone,Title
0,3,male,22.0,1,0,7.2500,S,2,0,Mr
1,1,female,38.0,1,0,71.2833,C,2,0,Mrs
2,3,female,26.0,0,0,7.9250,S,1,1,Miss
3,1,female,35.0,1,0,53.1000,S,2,0,Mrs
4,3,male,35.0,0,0,8.0500,S,1,1,Mr


In [3]:
X.isna().sum()[lambda s: s > 0]

Age         177
Embarked      2
dtype: int64

In [4]:
X.describe().T

,count,mean,std,min,25%,50%,75%,max
Pclass,891.0,2.308642,0.836071,1.00,2.0000,3.0000,3.0,3.0000
Age,714.0,29.699118,14.526497,0.42,20.1250,28.0000,38.0,80.0000
SibSp,891.0,0.523008,1.102743,0.00,0.0000,0.0000,1.0,8.0000
Parch,891.0,0.381594,0.806057,0.00,0.0000,0.0000,0.0,6.0000
Fare,891.0,32.204208,49.693429,0.00,7.9104,14.4542,31.0,512.3292
FamilySize,891.0,1.904602,1.613459,1.00,1.0000,1.0000,2.0,11.0000
IsAlone,891.0,0.602694,0.489615,0.00,0.0000,1.0000,1.0,1.0000


Missing Age (~20%) is imputed with the **train median** inside the pipeline. Cabin is dropped (77% missing). `Title`, `FamilySize`, `IsAlone` are engineered from Name/SibSp/Parch.

## California Housing (regression)

In [5]:
Xc, yc = load_california()
print(Xc.shape, 'target range', yc.min(), yc.max())
print('share of houses at the 5.0 cap: %.3f' % (yc >= 5.0).mean())
Xc.describe().T

(20640, 8) target range 0.14999 5.00001
share of houses at the 5.0 cap: 0.048


,count,mean,std,min,25%,50%,75%,max
MedInc,20640.0,3.870671,1.899822,0.499900,2.563400,3.534800,4.743250,15.000100
HouseAge,20640.0,28.639486,12.585558,1.000000,18.000000,29.000000,37.000000,52.000000
AveRooms,20640.0,5.429000,2.474173,0.846154,4.440716,5.229129,6.052381,141.909091
AveBedrms,20640.0,1.096675,0.473911,0.333333,1.006079,1.048780,1.099526,34.066667
Population,20640.0,1425.476744,1132.462122,3.000000,787.000000,1166.000000,1725.000000,35682.000000
AveOccup,20640.0,3.070655,10.386050,0.692308,2.429741,2.818116,3.282261,1243.333333
Latitude,20640.0,35.631861,2.135952,32.540000,33.930000,34.260000,37.710000,41.950000
Longitude,20640.0,-119.569704,2.003532,-124.350000,-121.800000,-118.490000,-118.010000,-114.310000


The target is capped at 5.0 ($500k): a ceiling effect that matters for error analysis later.

## Mall Customers (clustering)

In [6]:
m = load_mall(); print(m.shape); m.describe().T

(200, 5)


,count,mean,std,min,25%,50%,75%,max
CustomerID,200.0,100.50,57.879185,1.0,50.75,100.5,150.25,200.0
Age,200.0,38.85,13.969007,18.0,28.75,36.0,49.00,70.0
Annual Income (k$),200.0,60.56,26.264721,15.0,41.50,61.5,78.00,137.0
Spending Score (1-100),200.0,50.20,25.823522,1.0,34.75,50.0,73.00,99.0


200 rows, no labels. Clustering uses standardized Annual Income and Spending Score (Gender excluded: a binary one-hot distorts Euclidean distance).

## Leakage check: preprocessing is fitted on the training split only

In [7]:
Xtr, Xte, ytr, yte = split('clf', 0)
pre, n_num = build_preprocessor(Xtr); pre.fit(Xtr)
print('train/test sizes', len(Xtr), len(Xte), '| numeric cols first:', n_num)
print('scaler means come from TRAIN:', pre.named_transformers_['num'].named_steps['sc'].mean_.round(3))
print('test class share %.3f vs train %.3f (stratified)' % (yte.mean(), ytr.mean()))

train/test sizes 712 179 | numeric cols first: 7
scaler means come from TRAIN: [ 2.303 29.168  0.515  0.364 33.064  1.879  0.605]
test class share 0.385 vs train 0.383 (stratified)
